# Remote Pose Estimation Service (Colab GPU)

Runs **OpenPose** (BODY_25 → COCO-24) and **ROMP** (SMPL → COCO-24) on Colab's free GPU and exposes them as a FastAPI service over an ngrok tunnel. Your local FastAPI server calls this notebook's URL instead of running pose estimation on the laptop.

**Setup once per Colab session:**

1. **Runtime → Change runtime type → GPU (T4)** before running anything.
2. Get a free ngrok authtoken from <https://dashboard.ngrok.com/get-started/your-authtoken> and paste it into the cell labelled `NGROK_AUTHTOKEN`.
3. Run all cells top to bottom. The last cell prints a public URL like `https://abcd-1234.ngrok-free.app` — that's your remote pose API.
4. In your local `model/.env`, set `REMOTE_POSE_URL=<that url>` and restart the local FastAPI server.

**Endpoints exposed (async job queue):**

| Method | Path | Purpose |
|---|---|---|
| `GET`  | `/health` | Sanity check + `active_jobs` count |
| `POST` | `/pose/2d` | Multipart upload, returns `{job_id}` (202 Accepted) |
| `POST` | `/pose/3d` | Same, for 3D |
| `GET`  | `/jobs/{id}` | `{status, progress, error?, frame_count?}` |
| `GET`  | `/jobs/{id}/result` | ZIP of `frame_NNNNN.npz` files once status=done |
| `DELETE` | `/jobs/{id}` | Free server memory |

The async flow avoids ngrok's free-tier edge timeouts on long videos: a small POST returns immediately, the client polls a cheap status endpoint, then downloads the result in one final GET.

Colab idle-disconnects after ~90 minutes of no frontend activity and hard-disconnects after 12 hours. Section 10 installs a JS keepalive; if the runtime still dies, re-run all cells to get a fresh URL.

## 1. Verify GPU

In [ ]:
!nvidia-smi || echo 'No GPU â€” set Runtime > Change runtime type > GPU before continuing.'

## 2. Install dependencies

OpenPose can't be `pip install`ed cleanly. We install **`pytorch-openpose`**, a faithful BODY_25 PyTorch port that runs on GPU and produces compatible JSON-shaped output. ROMP installs straight from PyPI as `simple-romp` (same package your local service uses).

In [ ]:
%%capture
!pip install --quiet fastapi uvicorn pyngrok python-multipart
!pip install --quiet opencv-python-headless numpy scipy
!pip install --quiet simple-romp==1.1.3
# pytorch-openpose ships its modules under src/ rather than as a pip package,
# so we clone the repo and add it to sys.path at import time (cell 10).
!git clone --quiet https://github.com/Hzzone/pytorch-openpose.git /content/pytorch-openpose


## 3. Download OpenPose model weights

`pytorch-openpose` doesn't bundle weights. They live on a Google Drive published by the upstream author. We mirror the well-known direct link below.

In [ ]:
import pathlib, subprocess

MODEL_DIR = pathlib.Path("/content/openpose_models")
MODEL_DIR.mkdir(exist_ok=True)
BODY_WEIGHTS = MODEL_DIR / "body_pose_model.pth"

# pytorch-openpose body weights, mirrored on Hugging Face (no auth, no Drive permission games).
# Canonical ControlNet annotators mirror, identical to Hzzone's original release.
WEIGHTS_URL = "https://huggingface.co/lllyasviel/Annotators/resolve/main/body_pose_model.pth"

if not BODY_WEIGHTS.exists():
    subprocess.run(
        ["wget", "-q", "--show-progress", "-O", str(BODY_WEIGHTS), WEIGHTS_URL],
        check=True,
    )

size_mb = BODY_WEIGHTS.stat().st_size / 1e6
assert size_mb > 100, f"Weights file looks truncated ({size_mb:.1f} MB) - expected ~209 MB"
print(f"Body weights: {BODY_WEIGHTS} ({size_mb:.1f} MB)")


## 4. Pose inference helpers

These mirror the conversion logic in `model/services/pose_estimation_service.py` so the `.npz` files this notebook returns are byte-compatible with what your local `prediction_service` expects: `(1, 24, 2)` for 2D and `(1, 24, 3)` for 3D, COCO-24 ordering with hips at indices 8/11 and neck at 1.

In [ ]:
import numpy as np

def body18_to_coco24(kp18: np.ndarray) -> np.ndarray:
    """pytorch-openpose returns 18-keypoint COCO output (NOT BODY_25).
    Map it to the 24-keypoint COCO layout the local model was trained on.

    pytorch-openpose indices (18):
        0 Nose, 1 Neck, 2 RShoulder, 3 RElbow, 4 RWrist,
        5 LShoulder, 6 LElbow, 7 LWrist, 8 RHip, 9 RKnee, 10 RAnkle,
        11 LHip, 12 LKnee, 13 LAnkle, 14 REye, 15 LEye, 16 REar, 17 LEar.

    Missing/undetected keypoints come back as NaN; we coerce them to 0
    so they match the 'zero == missing' convention downstream.
    """
    coco24 = np.zeros((24, 2), dtype=np.float32)

    src = np.nan_to_num(kp18, nan=0.0)

    # Direct mapping for the first 18.
    coco24[:18] = src[:18]

    # 18 = Pelvis (mid-hip from L/R hip).
    if np.any(src[8]) and np.any(src[11]):
        coco24[18] = (src[8] + src[11]) / 2

    # 19 = Thorax (midpoint of neck and pelvis).
    if np.any(coco24[1]) and np.any(coco24[18]):
        coco24[19] = (coco24[1] + coco24[18]) / 2

    # 20 = Upper Neck (halfway between neck and nose).
    if np.any(coco24[1]) and np.any(coco24[0]):
        coco24[20] = coco24[1] + (coco24[0] - coco24[1]) * 0.5

    # 21 = Head Top (20px above nose, matching local converter).
    if np.any(coco24[0]):
        coco24[21] = np.array([coco24[0][0], max(0.0, coco24[0][1] - 20)])

    # 22, 23 = feet â€” pytorch-openpose has no foot keypoints, leave as zeros.
    return coco24


def smpl24_to_coco24(joints: np.ndarray) -> np.ndarray:
    """Same SMPLâ†’COCO24 mapping used by the local service for 3D."""
    out = np.zeros((24, 3), dtype=np.float32)
    try:
        out[0]  = joints[15]   # Nose <- Head
        out[1]  = joints[12]   # Neck
        out[2]  = joints[17]   # R Shoulder
        out[3]  = joints[19]   # R Elbow
        out[4]  = joints[21]   # R Wrist
        out[5]  = joints[16]   # L Shoulder
        out[6]  = joints[18]   # L Elbow
        out[7]  = joints[20]   # L Wrist
        out[8]  = joints[2]    # R Hip
        out[9]  = joints[5]    # R Knee
        out[10] = joints[8]    # R Ankle
        out[11] = joints[1]    # L Hip
        out[12] = joints[4]    # L Knee
        out[13] = joints[7]    # L Ankle
        out[14] = joints[15]   # R Eye
        out[15] = joints[15]   # L Eye
        out[16] = joints[15]   # R Ear
        out[17] = joints[15]   # L Ear
        out[18] = joints[0]    # Pelvis
        out[19] = joints[9]    # Thorax
        out[20] = joints[12]   # Upper Neck
        out[21] = joints[15]   # Head Top
        out[22] = joints[11]   # R Big Toe
        out[23] = joints[10]   # L Big Toe
    except IndexError:
        pass
    return out

## 5. Initialise the OpenPose and ROMP models (once)

Both models stay loaded across requests so we pay the load cost only at notebook startup, not per video.

In [ ]:
import os, sys, types, importlib.util

REPO = "/content/pytorch-openpose"
SRC  = os.path.join(REPO, "src")

# Sanity check the clone actually has the files we need.
for fname in ("body.py", "util.py", "model.py"):
    p = os.path.join(SRC, fname)
    assert os.path.isfile(p), f"missing {p} - rm -rf {REPO} and re-run cell 4"

# Wipe any half-loaded 'src' from previous failed cell runs.
for k in [k for k in sys.modules if k == "src" or k.startswith("src.")]:
    del sys.modules[k]

# Build a synthetic 'src' package and load body.py / util.py / model.py
# directly by file path, sidestepping sys.path entirely. This is robust to
# Colab's import caching, namespace shadowing, and missing __init__.py.
src_pkg = types.ModuleType("src")
src_pkg.__path__ = [SRC]
sys.modules["src"] = src_pkg

def _load(name):
    spec = importlib.util.spec_from_file_location(f"src.{name}", os.path.join(SRC, f"{name}.py"))
    mod  = importlib.util.module_from_spec(spec)
    sys.modules[f"src.{name}"] = mod
    setattr(src_pkg, name, mod)
    spec.loader.exec_module(mod)
    return mod

_load("util")
_load("model")
body_mod = _load("body")
Body = body_mod.Body

# Working directory must be the repo root - pytorch-openpose loads helper
# config files relative to CWD inside Body.__init__.
os.chdir(REPO)

import torch
import romp

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

openpose_body = Body(str(BODY_WEIGHTS))

# pytorch-openpose runs four input scales by default ([0.5, 1.0, 1.5, 2.0]).
# That's 4x forward passes plus 4x scipy gaussian_filter / NMS on CPU per frame
# and is the dominant cost. One scale at 0.5 keeps GPU fed and cuts wall time
# ~4x with negligible accuracy loss for full-body climbing footage.
try:
    body_mod.scale_search = [0.5]
except Exception:
    pass

print("OpenPose (pytorch-openpose) loaded")

romp_settings = romp.main.default_settings
romp_settings.mode = "image"
romp_settings.show_largest = True
# ROMP defaults GPU=-1 (CPU). Force GPU 0 when CUDA is available.
romp_settings.GPU = 0 if torch.cuda.is_available() else -1
romp_model = romp.ROMP(romp_settings)
print("ROMP loaded on GPU" if romp_settings.GPU >= 0 else "ROMP loaded on CPU")


## 6. Per-video processing functions

In [ ]:
import cv2, tempfile, shutil, zipfile, io, time
from pathlib import Path

def extract_frames(video_path: str, target_fps: int = 30) -> list:
    """Decode the video to a list of BGR frames sampled at target_fps."""
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        raise RuntimeError(f'Failed to open video: {video_path}')
    src_fps = cap.get(cv2.CAP_PROP_FPS) or target_fps
    step = max(1, int(round(src_fps / target_fps)))
    frames, idx = [], 0
    while True:
        ret, f = cap.read()
        if not ret:
            break
        if idx % step == 0:
            frames.append(f)
        idx += 1
    cap.release()
    return frames


def run_openpose_2d(frames: list) -> bytes:
    """Returns a zip (in memory) of frame_NNNNN.npz files, each (1, 24, 2)."""
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, frame in enumerate(frames):
            try:
                # pytorch-openpose's Body returns candidate (Nx4: x,y,score,id) and subset.
                # We just want the highest-scoring person's 18 keypoints.
                candidate, subset = openpose_body(frame)
                kp18 = np.zeros((18, 2), dtype=np.float32)
                if len(subset) > 0:
                    person = subset[0]  # already sorted by score in this fork
                    for j in range(18):
                        idx = int(person[j])
                        if idx >= 0:
                            kp18[j] = candidate[idx, :2]
                coords = body18_to_coco24(kp18)
            except Exception as e:
                print(f'frame {i}: {e}')
                coords = np.zeros((24, 2), dtype=np.float32)

            npz_buf = io.BytesIO()
            np.savez(npz_buf, coordinates=coords[np.newaxis, :, :])
            zf.writestr(f'frame_{i:05d}.npz', npz_buf.getvalue())
    buf.seek(0)
    return buf.read()


def run_romp_3d(frames: list) -> bytes:
    buf = io.BytesIO()
    with zipfile.ZipFile(buf, 'w', zipfile.ZIP_DEFLATED) as zf:
        for i, frame in enumerate(frames):
            try:
                results = romp_model(frame)
                joints = None
                if isinstance(results, dict):
                    joints = results.get('joints')
                if joints is None:
                    coords = np.zeros((24, 3), dtype=np.float32)
                else:
                    if joints.ndim == 3:
                        joints = joints[0]
                    coords = smpl24_to_coco24(joints[:24])
            except Exception as e:
                print(f'frame {i}: {e}')
                coords = np.zeros((24, 3), dtype=np.float32)

            npz_buf = io.BytesIO()
            np.savez(npz_buf, coordinates=coords[np.newaxis, :, :])
            zf.writestr(f'frame_{i:05d}.npz', npz_buf.getvalue())
    buf.seek(0)
    return buf.read()

## 7. FastAPI service — async job queue

Long pose-estimation requests (minutes per video) often die on ngrok's free edge proxy or any intermediate timeout. Instead of holding one HTTP request open, we run jobs in the background and let the client poll:

| Method | Path | Purpose |
|---|---|---|
| `POST` | `/pose/2d` | Enqueue a 2D job, returns `{job_id}` immediately |
| `POST` | `/pose/3d` | Enqueue a 3D job, returns `{job_id}` |
| `GET`  | `/jobs/{job_id}` | Returns `{status: queued\|running\|done\|error, progress, error?}` |
| `GET`  | `/jobs/{job_id}/result` | Streams the ZIP once `status=done` |

Uvicorn runs in a background thread (Colab can't bind directly).

In [ ]:
from fastapi import FastAPI, UploadFile, File, HTTPException, BackgroundTasks
from fastapi.responses import Response, JSONResponse
import threading, uvicorn, tempfile, os, uuid

app = FastAPI(title='Remote Pose Estimation Service')

# In-memory job table. Colab is single-process so a dict is fine; a Lock
# guards against the FastAPI event loop racing the worker thread.
JOBS = {}
JOBS_LOCK = threading.Lock()
JOB_QUEUE = []           # list of job_ids waiting to run
QUEUE_COND = threading.Condition()

def _set(job_id, **kw):
    with JOBS_LOCK:
        JOBS[job_id].update(kw)

def _worker():
    """Single worker thread. ROMP/OpenPose are not thread-safe for parallel
    inference on one GPU, so we serialise jobs."""
    while True:
        with QUEUE_COND:
            while not JOB_QUEUE:
                QUEUE_COND.wait()
            job_id = JOB_QUEUE.pop(0)
        job = JOBS.get(job_id)
        if job is None:
            continue
        try:
            _set(job_id, status='running', progress=0.0)
            frames = extract_frames(job['video_path'], target_fps=job['fps'])
            if not frames:
                raise RuntimeError('No frames decoded from video')
            _set(job_id, frame_count=len(frames))
            runner = run_openpose_2d if job['kind'] == '2d' else run_romp_3d
            zip_bytes = runner(frames)
            _set(job_id, status='done', progress=1.0, result=zip_bytes)
        except Exception as e:
            _set(job_id, status='error', error=str(e))
        finally:
            try:
                os.unlink(job['video_path'])
            except OSError:
                pass

threading.Thread(target=_worker, daemon=True).start()


@app.get('/health')
def health():
    with JOBS_LOCK:
        active = sum(1 for j in JOBS.values() if j['status'] in ('queued', 'running'))
    return {'ok': True, 'gpu': torch.cuda.is_available(), 'device': device, 'active_jobs': active}


async def _save_upload(video: UploadFile) -> str:
    suffix = Path(video.filename).suffix or '.mp4'
    fd, path = tempfile.mkstemp(suffix=suffix)
    with os.fdopen(fd, 'wb') as f:
        f.write(await video.read())
    return path


def _enqueue(kind: str, video_path: str, fps: int) -> str:
    job_id = uuid.uuid4().hex
    with JOBS_LOCK:
        JOBS[job_id] = {
            'status': 'queued', 'kind': kind, 'video_path': video_path,
            'fps': fps, 'progress': 0.0, 'created': time.time(),
        }
    with QUEUE_COND:
        JOB_QUEUE.append(job_id)
        QUEUE_COND.notify()
    return job_id


@app.post('/pose/2d')
async def pose_2d(video: UploadFile = File(...), fps: int = 30):
    path = await _save_upload(video)
    job_id = _enqueue('2d', path, fps)
    return JSONResponse({'job_id': job_id, 'status': 'queued'}, status_code=202)


@app.post('/pose/3d')
async def pose_3d(video: UploadFile = File(...), fps: int = 30):
    path = await _save_upload(video)
    job_id = _enqueue('3d', path, fps)
    return JSONResponse({'job_id': job_id, 'status': 'queued'}, status_code=202)


@app.get('/jobs/{job_id}')
def job_status(job_id: str):
    with JOBS_LOCK:
        job = JOBS.get(job_id)
        if job is None:
            raise HTTPException(404, 'Unknown job')
        # Don't leak the raw zip bytes through the status endpoint.
        return {k: v for k, v in job.items() if k not in ('result', 'video_path')}


@app.get('/jobs/{job_id}/result')
def job_result(job_id: str):
    with JOBS_LOCK:
        job = JOBS.get(job_id)
        if job is None:
            raise HTTPException(404, 'Unknown job')
        if job['status'] != 'done':
            raise HTTPException(409, f"Job not done (status={job['status']})")
        zip_bytes = job['result']
        frames = job.get('frame_count', 0)
    return Response(zip_bytes, media_type='application/zip', headers={
        'X-Frame-Count': str(frames),
    })


@app.delete('/jobs/{job_id}')
def job_delete(job_id: str):
    with JOBS_LOCK:
        JOBS.pop(job_id, None)
    return {'ok': True}


def _run_uvicorn():
    uvicorn.run(app, host='0.0.0.0', port=8000, log_level='warning')

thread = threading.Thread(target=_run_uvicorn, daemon=True)
thread.start()
time.sleep(2)
print('FastAPI listening on :8000')

## 8. Expose via ngrok

Paste your free authtoken below.

In [ ]:
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = ''  # <-- paste from https://dashboard.ngrok.com/get-started/your-authtoken

if not NGROK_AUTHTOKEN:
    raise SystemExit('Set NGROK_AUTHTOKEN above before running this cell.')

conf.get_default().auth_token = NGROK_AUTHTOKEN
for t in ngrok.get_tunnels():
    ngrok.disconnect(t.public_url)

tunnel = ngrok.connect(8000, 'http')
print('=' * 60)
print('Public URL :', tunnel.public_url)
print('Health check:', tunnel.public_url + '/health')
print('=' * 60)
print()
print('In your local model/.env, set:')
print(f'  REMOTE_POSE_URL={tunnel.public_url}')
print('Then restart the local FastAPI server.')

## 9. Optional: smoke-test from inside Colab

Upload a short test video to `/content/` (left sidebar â†’ Files â†’ Upload) named `test.mp4`, then run this cell to confirm both endpoints return non-empty zips.

In [ ]:
import requests, io, zipfile, glob

video_path = '/content/test.mp4'
if not os.path.exists(video_path):
    print('Upload /content/test.mp4 first to run this smoke test.')
else:
    for endpoint in ['/pose/2d', '/pose/3d']:
        with open(video_path, 'rb') as f:
            r = requests.post(tunnel.public_url + endpoint, files={'video': f}, timeout=60)
        r.raise_for_status()
        job_id = r.json()['job_id']
        print(endpoint, '-> job', job_id)
        # Poll until done.
        while True:
            s = requests.get(f'{tunnel.public_url}/jobs/{job_id}', timeout=30).json()
            print(' ', s['status'], 'progress=', s.get('progress'))
            if s['status'] in ('done', 'error'):
                break
            time.sleep(2)
        if s['status'] == 'error':
            print('  ERROR:', s.get('error'))
            continue
        rr = requests.get(f'{tunnel.public_url}/jobs/{job_id}/result', timeout=120)
        print('  frames:', rr.headers.get('X-Frame-Count'), 'bytes:', len(rr.content))
        zf = zipfile.ZipFile(io.BytesIO(rr.content))
        sample = zf.namelist()[0]
        with zf.open(sample) as fh:
            arr = np.load(io.BytesIO(fh.read()))['coordinates']
        print(f'  {sample}: shape={arr.shape}, nonzero={int((arr != 0).sum())}')

## 10. Keep the notebook awake

Colab disconnects the runtime when the browser tab loses frontend activity — `time.sleep` in Python doesn't count. The cell below injects a JS snippet that clicks the toolbar's "connect" button every minute, which is what real users do to stay alive. Run it once and leave the tab open (it can be backgrounded).

The Python `while True` underneath is there too so the cell stays "running" in Colab's eyes (busy cells are also considered active). Stop it (■) when you're done.

In [ ]:
from IPython.display import display, Javascript
import time

# JS keepalive: every 60s, find the Colab "connect" button and click it.
# Selectors differ slightly by Colab UI version, so we try a few.
display(Javascript('''
function colabKeepAlive() {
  const sels = [
    'colab-toolbar-button#connect',
    'colab-connect-button',
    '#top-toolbar > colab-connect-button',
  ];
  for (const sel of sels) {
    const el = document.querySelector(sel);
    if (el) { el.click(); console.log('keepalive click', sel); return; }
  }
  console.log('keepalive: no connect button found');
}
if (window.__colabKeepAliveTimer) clearInterval(window.__colabKeepAliveTimer);
window.__colabKeepAliveTimer = setInterval(colabKeepAlive, 60 * 1000);
console.log('Colab keepalive installed (every 60s).');
'''))

print('Service is live. Keepalive JS installed. Press the stop button on this cell to shut down.')
while True:
    time.sleep(60)
    # Print a heartbeat so the cell registers stdout activity periodically.
    print('.', end='', flush=True)